# Imports and Configs

In [ ]:
# Install klib and flash libraries
!pip install klib git+https://github.com/flash-lib/flash.git -q

In [ ]:
# Standard Libraries
import toml

# Data Manipulation
import numpy as np
import pandas as pd

# Data Analysis
import klib
import flash as fz
import seaborn as sns
import plotly.graph_objs as go
import matplotlib.pyplot as plt

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

In [ ]:
# Load the configurations
with open("config/config.toml", "r") as file:
    config_data = toml.load(file)

num_cols, cat_cols = config_data['num']['cols'], config_data['cat']['cols']

# Preparation

In [ ]:
# Load the dataset
df = pd.read_csv('data/interim/cleaned_train_data_v2.csv')

In [ ]:
# Convert categorical features' data type to category
# This will be helpful while doing analysis
df[cat_cols] = df[cat_cols].astype('category')

# Test
df.dtypes

# EDA on features

## Univariate analysis

### Numerical

In [ ]:
# Statistical measures
df[num_cols].describe().T

In [ ]:
# Statistical moments
fz.stats_moments(df[num_cols])

In [ ]:
# Plotting histogram & boxplot
fig, axs = fz.hist_box_viz(df[num_cols])
fig

#### Conclusions:

- There are many outliers on the upper side of all numerical features, while none are present on the lower side.
- Since the outliers appear to be valid and are not due to data entry issues, we don't have to drop them.
- None of the numerical features follow a normal distribution.
- The distributions of applicant income and loan amount are right-skewed (positively skewed).
- Feature transformation is required for all numerical features to address this skewness.
- It looks like people with a co-applicant income of 0 doesn't have a co-applicant. So, we should create a new feature called 'has_coapplicant'. For this feature, set the value to 'no' for individuals with a co-applicant income of 0, and 'yes' for those with a non-zero co-applicant income.

### Categorical

In [ ]:
# Statistical measures
df[cat_cols].describe().T

In [ ]:
# Countplots
fig, axs = fz.count_viz(df[cat_cols])
fig

#### Conclusions:

- A higher number of males apply for loans compared to females.
- Married individuals are more likely to apply for loans than unmarried individuals, with approximately twice as many married applicants.
- Individuals without dependents apply for loans more frequently than those with dependents.
- Graduates are more likely to apply for loans than non-graduates.
- Non-self-employed individuals apply for loans more than self-employed individuals.
- People whose property is located in semi-urban areas tend to apply for loans more than those with properties in rural or urban areas. Those with property in rural areas apply for the fewest loans, although these trends are not very strong.
- The majority of loan applicants prefer a loan term of 360 months (30 years), followed by 180 months (15 years). Other loan term durations are relatively rare.
- Individuals with a credit history of 1 are more likely to apply for loans compared to those with a credit history of 0.

## Bivariate analysis

### Numerical - Numerical

In [ ]:
# Pairplot
grid = fz.pair_viz(df[num_cols])
plt.show()

In [ ]:
# Correlation heatmap
methods=['pearson', 'spearman', 'kendall']
for method in methods:
    fz.corr_heatmap_viz(df[num_cols], method=method)
    plt.show()
    print("-"*150)

Conclusions:

- None of the features show a strong linear relationship with each other. However, there is a moderate relationship between applicant income and loan amount. This makes sense because individuals with higher incomes often need larger loan amounts.

- Pearson, Spearman, and Kendall
Tau's correlations show similar patterns, but their values are slightly different. Since the heatmaps from all of these are similar, the exact values are less important. In this case, Spearman's correlation is more suitable because the data isn't normally distributed, doesn't have a linear relationship between features, and has outliers.

### Categorical - Categorical

In [ ]:
# Crosstab Heatmap
fz.crosstab_heatmap_viz(df, cat_cols, normalize='both')

### Numerical - Categorical

#### Box-plot

In [ ]:
# applicant_income
fig, axs = fz.num_cat_viz(df, 'applicant_income', cat_cols)
fig

In [ ]:
# coapplicant_income
fig, axs = fz.num_cat_viz(df, 'coapplicant_income', cat_cols)
fig

In [ ]:
# loan_amount
fig, axs = fz.num_cat_viz(df, 'loan_amount', cat_cols)
fig

#### KDE-plot

In [ ]:
# applicant_income
fig, axs = fz.num_cat_viz(df, 'applicant_income', cat_cols, kind='kde')
fig

In [ ]:
# coapplicant_income
fig, axs = fz.num_cat_viz(df, 'coapplicant_income', cat_cols, kind='kde')
fig

In [ ]:
# loan_amount
fig, axs = fz.num_cat_viz(df, 'loan_amount', cat_cols, kind='kde')
fig

#### Point-plot

In [ ]:
# applicant_income
fig, axs = fz.num_cat_viz(df, 'applicant_income', cat_cols, kind='point')
fig

In [ ]:
# coapplicant_income
fig, axs = fz.num_cat_viz(df, 'coapplicant_income', cat_cols, kind='point')
fig

In [ ]:
# loan_amount
fig, axs = fz.num_cat_viz(df, 'loan_amount', cat_cols, kind='point')
fig

## Mulitvariate analysis

### Numerical - Numerical - Numerical

In [ ]:
trace = go.Scatter3d(
    x=df['applicant_income'],
    y=df['coapplicant_income'],
    z=df['loan_amount'],
    mode='markers',
    marker=dict(size=5)
)

layout = go.Layout(
    scene=dict(
        xaxis_title='Applicant Income',
        yaxis_title='Coapplicant Income',
        zaxis_title='Loan Amount'
    )
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

### Numerical - Numerical - Categorical

In [ ]:
# Pairplot
for feature in cat_cols:
    sns.pairplot(df, vars=num_cols, hue=feature)
    plt.show()
    print("-"*105)

In [ ]:
def num_num_cat_viz(x, y, categorical_features):
    for feature in categorical_features:
        sns.relplot(df, x=x, y=y, col=feature)
        plt.show()
        print("-"*118)

In [ ]:
# applicant_income & coapplicant_income
num_num_cat_viz('applicant_income', 'coapplicant_income', cat_cols)

In [ ]:
# applicant_income & loan_amount
num_num_cat_viz('applicant_income', 'loan_amount', cat_cols)

In [ ]:
# coapplicant_income & loan_amount
num_num_cat_viz('coapplicant_income', 'loan_amount', cat_cols)

### Numerical - Categorical - Categorical

# EDA on target

## Univariate analysis

In [ ]:
plt.pie(df['loan_status'].value_counts(), labels=df['loan_status'].unique(), autopct='%0.2f%%',
        shadow=True, explode=(0, 0.1), counterclock=False, colors=['lime', 'cyan'])
plt.show()

#### Conclusions:

- The classes in target is moderately imbalanced. Need to handle the class imbalance using SMOTE.

## Bivariate analysis


### Categorical - Categorical


In [ ]:
# Crosstab Heatmap
fz.crosstab_heatmap_viz(df, cat_cols, ['loan_status'], 'both')


### Numerical - Categorical


In [ ]:
# Box-plot
fig, axs = fz.num_cat_viz(df, num_cols, 'loan_status')
fig

In [ ]:
# KDE-plot
fig, axs = fz.num_cat_viz(df, num_cols, 'loan_status', kind='kde')
fig

In [ ]:
# Point-plot
fig, axs = fz.num_cat_viz(df, num_cols, 'loan_status', kind='point')
fig

## Mulitvariate analysis


### Numerical - Numerical - Categorical

In [ ]:
sns.pairplot(df, vars = num_cols, hue='loan_status')

In [ ]:
def relplot(df, numerical_features, categorical_feature):
    for i, feature_i in enumerate(numerical_features):
        for j, feature_j in enumerate(numerical_features[i+1:], start=i+1):
            sns.relplot(df, x=feature_i, y=feature_j, col=categorical_feature)
            plt.show()
            print("-" * df[categorical_feature].nunique()*59)

In [ ]:
relplot(df, num_cols, 'loan_status')


### Numerical - Categorical - Categorical